# L100 · Give an Agent Short Term Memory with Lakebase

A single call agent forgets everything between questions. Real assistants remember the conversation. This notebook gives an agent short term memory by storing each turn in Lakebase, the managed Postgres database on Databricks, keyed by a thread id.

Lakebase is built for this: low latency reads and writes, so an agent can fetch a thread's history on every turn without the cost of a full table scan.

### What you will learn
- How to connect to a Lakebase instance from a notebook using a short lived credential.
- How to store and retrieve conversation turns by thread id.
- How memory lets the agent answer a follow up that depends on an earlier turn.
- How separate threads stay isolated.

> Note: all content is synthetic, for the workshop.


## 0. Install dependencies

`psycopg` is the Postgres driver. The Databricks SDK mints a short lived database credential so you never handle a long lived password.


In [ ]:
%pip install -qq "psycopg[binary]" databricks-sdk --upgrade
dbutils.library.restartPython()

## 1. Connect to Lakebase

The setup cell reads the widgets, mints a credential for the Lakebase instance, and opens a connection. The token is short lived and scoped to your identity, which is the secure pattern: no static secret in the notebook.


In [ ]:
import re, uuid
import psycopg
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "LLM endpoint")
dbutils.widgets.text("lakebase_instance", "prism-lakebase", "Lakebase instance name")
dbutils.widgets.text("database", "databricks_postgres", "Postgres database")

llm_endpoint = dbutils.widgets.get("llm_endpoint")
lakebase_instance = dbutils.widgets.get("lakebase_instance")
database = dbutils.widgets.get("database")
for name, val in [("llm_endpoint", llm_endpoint), ("lakebase_instance", lakebase_instance), ("database", database)]:
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", val):
        raise ValueError(f"Unsafe {name}: {val!r}.")

w = WorkspaceClient()
instance = w.database.get_database_instance(name=lakebase_instance)
cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()), instance_names=[lakebase_instance]
)
pg_user = w.current_user.me().user_name

conn = psycopg.connect(
    host=instance.read_write_dns,
    port=5432,
    dbname=database,
    user=pg_user,
    password=cred.token,
    sslmode="require",
    autocommit=True,
)
print(f"Connected to Lakebase instance {lakebase_instance} as {pg_user}")

## 2. Create the thread store

One table holds every conversation turn. The `thread_id` groups turns into a conversation, `turn` orders them, and `role` marks who spoke. A primary key on `(thread_id, turn)` prevents duplicate turns. This demo writes one turn at a time per thread, which is fine here; a production agent with concurrent writers on the same thread would use a database sequence or an `ON CONFLICT` clause.


In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS akzo_agent_memory (
            thread_id  TEXT        NOT NULL,
            turn       INT         NOT NULL,
            role       TEXT        NOT NULL,
            content    TEXT        NOT NULL,
            created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
            PRIMARY KEY (thread_id, turn)
        )
    """)
print("Table akzo_agent_memory is ready.")

## 3. Build the agent with memory

`chat` is the agent. On every call it loads the thread's prior turns from Lakebase, sends them with the new question to the Foundation Model, then writes the new user and assistant turns back. The thread id is the only state the caller needs to pass.


In [ ]:
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

SYSTEM_PROMPT = "You are an assistant for AkzoNobel. Use the conversation so far to answer. Be concise."
_ROLE = {"system": ChatMessageRole.SYSTEM, "user": ChatMessageRole.USER, "assistant": ChatMessageRole.ASSISTANT}

def _load_history(thread_id):
    with conn.cursor() as cur:
        cur.execute(
            "SELECT role, content FROM akzo_agent_memory WHERE thread_id = %s ORDER BY turn",
            (thread_id,),
        )
        return [{"role": r, "content": c} for r, c in cur.fetchall()]

def _next_turn(thread_id):
    with conn.cursor() as cur:
        cur.execute("SELECT COALESCE(MAX(turn), -1) + 1 FROM akzo_agent_memory WHERE thread_id = %s", (thread_id,))
        return cur.fetchone()[0]

def _save(thread_id, turn, role, content):
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO akzo_agent_memory (thread_id, turn, role, content) VALUES (%s, %s, %s, %s)",
            (thread_id, turn, role, content),
        )

def chat(thread_id: str, user_msg: str) -> str:
    history = _load_history(thread_id)
    messages = [ChatMessage(role=ChatMessageRole.SYSTEM, content=SYSTEM_PROMPT)]
    messages += [ChatMessage(role=_ROLE[h["role"]], content=h["content"]) for h in history]
    messages.append(ChatMessage(role=ChatMessageRole.USER, content=user_msg))
    reply = w.serving_endpoints.query(name=llm_endpoint, messages=messages, max_tokens=256, temperature=0.0)
    answer = reply.choices[0].message.content
    # Write the user and assistant turns atomically, so a failure cannot leave a
    # dangling user turn with no reply.
    with conn.transaction():
        t = _next_turn(thread_id)
        _save(thread_id, t, "user", user_msg)
        _save(thread_id, t + 1, "assistant", answer)
    return answer

print("Agent ready.")

## 4. Prove the memory works

Use a fresh thread id. Turn one states a fact. Turn two asks a question that can only be answered by remembering turn one. If memory works, the agent recalls EMEA without being told again.


In [ ]:
thread_a = f"demo-a-{uuid.uuid4().hex[:8]}"

print("Turn 1:")
print(chat(thread_a, "Hi, my name is Priya and I manage the EMEA finance controlling team."))
print("\nTurn 2:")
print(chat(thread_a, "Which region do I manage? Answer in one short sentence."))

## 5. Threads stay isolated

A different thread id is a different conversation. The agent should not know Priya or EMEA here, which shows memory is scoped to the thread, not global.


In [ ]:
thread_b = f"demo-b-{uuid.uuid4().hex[:8]}"
print(chat(thread_b, "What region do I manage? If you do not know, say so."))

## 6. Look at the stored turns

The conversation lives in Lakebase, durable and queryable. This is the same store an agent in production would read on every request.


In [ ]:
import pandas as pd
with conn.cursor() as cur:
    cur.execute(
        "SELECT thread_id, turn, role, content FROM akzo_agent_memory WHERE thread_id IN (%s, %s) ORDER BY thread_id, turn",
        (thread_a, thread_b),
    )
    rows = cur.fetchall()
display(pd.DataFrame(rows, columns=["thread_id", "turn", "role", "content"]))

## 7. Close the connection

Always release the database connection when you are done, so the pool is not held open.


In [ ]:
conn.close()
print("Lakebase connection closed.")

## Wrap up

You gave an agent memory backed by Lakebase, with thread scoped, durable, low latency state.

| Tier | Memory |
| --- | --- |
| L100 (here) | Short term memory: a thread store in Lakebase that the agent reads each turn |
| L200 | The same store wired into a tool calling agent through the Agent Framework |
| L300 | Per user, governed memory across the multi domain supervisor |

This table stores raw conversation text. It holds synthetic workshop data here. In production you would add access controls, a retention policy, and per user scoping before storing real conversations.

### Next
You finished the L100 notebooks. Open `01_agent_bricks_types.md` to build the no code Agent Bricks agents, then move up to `../L200-capabilities/`.
